# Mission 4 · 가설 검정용 Baseline CNN 5-fold

Mission 0의 가설을 검정하려면 Baseline도 최종 모델과 같은 fold 분할로 OOF 예측이 있어야 한다. 해커톤 제공 Baseline CNN을 그대로 두고 5-fold로 학습한다.

* **B5**: 원래 Baseline 전처리 (흑백, 128×128 강제 Resize)
* **B5L**: 전처리만 letterbox(비율 유지 후 128×128)로 바꾼 버전

train 이미지는 이미 224×224 정사각형 letterbox라서 두 전처리가 train에서는 같은 결과를 낸다. 차이는 원본 비율이 제각각인 test에서만 생긴다. 따라서 가설 1(전처리)은 리더보드로, 가설 2(모델과 학습)는 OOF McNemar 검정으로 확인한다.

## 1. 환경 설정과 데이터 로드

데이터 폴더는 `open (1)` 또는 `../open (1)`에서 자동으로 찾는다. fold 분할은 다른 실험과 같은 `StratifiedKFold(5, shuffle=True, random_state=42)`이다.

In [1]:
import os, random, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

D = next(p for p in ["../../data"] if os.path.exists(os.path.join(p, "train.csv")))
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
OUT = "../../outputs"; os.makedirs(OUT, exist_ok=True)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

tr = pd.read_csv(os.path.join(D, "train.csv")); te = pd.read_csv(os.path.join(D, "test.csv"))
y = tr.label.astype(int).values
train_raw = [Image.open(os.path.join(D, "train", f)).convert("L") for f in tr.file_name]
test_raw = [Image.open(os.path.join(D, "test", f)).convert("L") for f in te.file_name]
splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
print("device:", device, "| train", len(tr), "| test", len(te))

device: mps | train 5216 | test 624


## 2. 두 가지 전처리

B5는 원래 Baseline처럼 128×128로 강제 Resize한다. B5L은 비율을 유지해 긴 변을 128에 맞추고 나머지를 검은색으로 채운다. 정규화와 증강은 원래 Baseline처럼 쓰지 않는다.

In [2]:
def letterbox(img, size=128):
    w, h = img.size; s = size / max(w, h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    c = Image.new("L", (size, size), 0); c.paste(img.resize((nw, nh), Image.BILINEAR), ((size - nw) // 2, (size - nh) // 2)); return c
to_tensor = transforms.ToTensor()
PREP = {"B5": lambda im: to_tensor(im.resize((128, 128), Image.BILINEAR)),
        "B5L": lambda im: to_tensor(letterbox(im, 128))}

# train에서 두 전처리가 같은지 확인 (train은 이미 정사각형)
diff = max(float((PREP["B5"](train_raw[i]) - PREP["B5L"](train_raw[i])).abs().max()) for i in range(20))
print("train 20장에서 두 전처리 결과의 최대 차이:", round(diff, 6))

train 20장에서 두 전처리 결과의 최대 차이: 0.0


## 3. Baseline CNN과 학습

해커톤 제공 Baseline과 같은 구조와 설정이다. 합성곱 2층, Adam(lr 1e-3), batch 32, 2 epoch, 가중치 없는 교차 엔트로피.

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                  nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(32 * 32 * 32, 2))
    def forward(self, x): return self.fc(self.conv(x))

class TensorDS(Dataset):
    def __init__(self, X, Y=None): self.X, self.Y = X, Y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return (self.X[i], int(self.Y[i])) if self.Y is not None else self.X[i]

@torch.no_grad()
def predict(m, X):
    m.eval(); return torch.cat([F.softmax(m(x.to(device)), 1)[:, 1].cpu() for x in DataLoader(TensorDS(X), batch_size=128)]).numpy()

def run(tag):
    Xtr = torch.stack([PREP[tag](im) for im in train_raw]); Xte = torch.stack([PREP[tag](im) for im in test_raw])
    oof = np.zeros(len(y)); test_probs = []
    for k, (tr_idx, va_idx) in enumerate(splits):
        seed_all(42 + k)
        m = SimpleCNN().to(device); opt = torch.optim.Adam(m.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
        g = torch.Generator(); g.manual_seed(42 + k)
        loader = DataLoader(TensorDS(Xtr[tr_idx], y[tr_idx]), batch_size=32, shuffle=True, generator=g)
        for ep in range(2):
            m.train()
            for xb, yb in loader:
                loss = crit(m(xb.to(device)), yb.to(device)); opt.zero_grad(); loss.backward(); opt.step()
        oof[va_idx] = predict(m, Xtr[va_idx]); test_probs.append(predict(m, Xte))
        print(f"{tag} fold {k}: val acc {accuracy_score(y[va_idx], oof[va_idx] >= 0.5):.4f}")
    np.save(f"{OUT}/oof_{tag}.npy", oof)
    tp = np.mean(test_probs, axis=0); np.save(f"{OUT}/test_prob_{tag}.npy", tp)
    sub = te.copy(); sub["label"] = (tp >= 0.5).astype(int); sub.to_csv(f"{OUT}/submission_baselinecnn_{tag}_f01234.csv", index=False)
    pr = (oof >= 0.5).astype(int); tn, fp, fn, tpn = confusion_matrix(y, pr).ravel()
    print(f"{tag} OOF acc {accuracy_score(y, pr):.4f} | AUC {roc_auc_score(y, oof):.4f} | FN {fn} FP {fp} | test NORMAL {int((sub.label==0).sum())}")
    return oof, sub

oof_b5, sub_b5 = run("B5")
oof_b5l, sub_b5l = run("B5L")
print("test 예측이 다른 장수 (B5 vs B5L):", int((sub_b5.label != sub_b5l.label).sum()))

B5 fold 0: val acc 0.9540


B5 fold 1: val acc 0.9597


B5 fold 2: val acc 0.9559


B5 fold 3: val acc 0.9569


B5 fold 4: val acc 0.9271
B5 OOF acc 0.9507 | AUC 0.9868 | FN 141 FP 116 | test NORMAL 311


B5L fold 0: val acc 0.9540


B5L fold 1: val acc 0.9597


B5L fold 2: val acc 0.9559


B5L fold 3: val acc 0.9569


B5L fold 4: val acc 0.9271
B5L OOF acc 0.9507 | AUC 0.9868 | FN 141 FP 116 | test NORMAL 105
test 예측이 다른 장수 (B5 vs B5L): 206
